In [11]:
import pandas as pd
import ast

# carregar dataset
df = pd.read_csv("../data/raw/pii_dataset.csv")

# converter string de lista para lista real
df["labels"] = df["labels"].apply(ast.literal_eval)

# função para detectar PII
def detect_pii(labels):
    for l in labels:
        if l != "O":
            return 1
    return 0

# criar coluna de classificação
df["label"] = df["labels"].apply(detect_pii)

# manter apenas texto e label
classification_df = df[["text", "label"]]

# salvar dataset convertido
classification_df.to_csv(
    "../data/raw/pii_classification_dataset.csv",
    index=False
)

print("Dataset convertido criado em:")
print("data/raw/pii_classification_dataset.csv")

Dataset convertido criado em:
data/raw/pii_classification_dataset.csv


In [12]:
classification_df['label'].value_counts()

label
1    4425
0       9
Name: count, dtype: int64

In [13]:
import pandas as pd
import ast
import re

df = pd.read_csv("../data/raw/pii_dataset.csv")

df["labels"] = df["labels"].apply(ast.literal_eval)

def has_pii(labels):
    return any(label != "O" for label in labels)

df["label"] = df["labels"].apply(has_pii).astype(int)

pii_df = df[df["label"] == 1].copy()

def remove_pii(text):

    text = re.sub(r'\S+@\S+', '[EMAIL]', text)
    text = re.sub(r'\d{10,}', '[PHONE]', text)
    text = re.sub(r'\d{3}\.\d{3}\.\d{3}-\d{2}', '[CPF]', text)

    return text

non_pii_df = pii_df.copy()
non_pii_df["text"] = non_pii_df["text"].apply(remove_pii)
non_pii_df["label"] = 0

balanced_df = pd.concat([
    pii_df.sample(1000, random_state=42),
    non_pii_df.sample(1000, random_state=42)
])

balanced_df = balanced_df[["text", "label"]]

balanced_df.to_csv(
    "../data/raw/pii_classification_balanced.csv",
    index=False
)

print("Dataset balanceado criado.")
print(balanced_df["label"].value_counts())

Dataset balanceado criado.
label
1    1000
0    1000
Name: count, dtype: int64


In [ ]:
import pandas as pd
import re

df = pd.read_csv("../data/raw/pii_dataset.csv")

def clean_text(text):

    text = text.lower()
    text = re.sub(r'[^a-z0-9\s@.]', '', text)
    text = re.sub(r'\s+', ' ', text)

    return text

df["text"] = df["text"].apply(clean_text)

df.to_csv(".data/processed/clean_text.csv", index=False)